In [ ]:
import pyxdf

# for the tests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import SpanSelector

%matplotlib qt

In [ ]:
def print_streams_types_and_names(fullFname_or_streams):
    """Print the names and types of all streams in the xdf file or in the streams list"""

    if isinstance(fullFname_or_streams, str):
        xdf_data, header = pyxdf.load_xdf(filename=fullFname_or_streams, verbose=False)
    elif (
        isinstance(fullFname_or_streams, list)
        and all(isinstance(x, dict) for x in fullFname_or_streams)
        and all("info" in x for x in fullFname_or_streams)
    ):
        xdf_data = fullFname_or_streams
    else:
        raise ValueError("The first argument must be a filename or a list of streams")

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, searched_stream_type, searched_stream_names):
    """Get the stream of type 'searched_stream_type' with name in 'searched_stream_names' in the xdf_data"""

    if not isinstance(
        searched_stream_names, list
    ):  # if we get a string (only one name)
        searched_stream_names = [searched_stream_names]

    found_streams = []
    for stream in xdf_data:
        stream_type = stream["info"]["type"][0]
        if searched_stream_type == stream_type:
            stream_name = stream["info"]["name"][0]
            for searched_stream_name in searched_stream_names:
                if searched_stream_name == stream_name:
                    found_streams.append(stream)

    if not found_streams:
        # msg = f" Stream not found. Searched in [{searched_stream_type}:{searched_stream_names}]."
        # print(msg)
        return None

    if len(found_streams) > 1:
        found_streams_names = [stream["info"]["name"][0] for stream in found_streams]
        msg = f"Found multiple streams: [{searched_stream_type},{found_streams_names}]."
        raise ValueError(msg)

    return found_streams[0]


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3/ReArm_C1P02_20210715_V3_Reaching/ReArm_C1P07_20211116_V3_r.xdf"


xdf_data, header = pyxdf.load_xdf(
    filename=xdf_fullFname,
    select_streams=[
        {"type": "MoCap"},
        {"type": "Markers"},
    ],
    synchronize_clocks=True,
    dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    verbose=False,
)

print_streams_types_and_names(xdf_data)

In [ ]:
# read the time correction file
time_correction_file = xdf_fullFname.replace(".xdf", "_xdf_time_correction.csv")
time_correction = np.loadtxt(time_correction_file, delimiter=",", skiprows=1)
# print the time correction
print(f"Time correction: {time_correction} s")

# make the time correction
kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

if kinect_mocap:
    kinect_mocap["time_stamps"] = kinect_mocap["time_stamps"] + time_correction
if kinect_markers:
    kinect_markers["time_stamps"] = kinect_markers["time_stamps"] + time_correction

In [ ]:
def find__mouse_marker_indexes(marker_name, markers_data):
    """Find the indexes of the marker in the markers data"""

    marker_index_list = [
        i for i, marker in enumerate(markers_data) if marker_name in marker[0]
    ]

    return marker_index_list


def get_start_stop_times_from_mouse_to_nic_markers(mouse_to_nic_markers):
    """get the start and stop of the mouse motion from MouseToNIC"""

    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]

    if not isinstance(mouse_to_nic_markers_data[0], list):
        mouse_to_nic_markers_data = [[str(x)] for x in mouse_to_nic_markers_data]

    start_marker_index_list = find__mouse_marker_indexes(
        "[111]", mouse_to_nic_markers_data
    )
    stop_marker_index_list = find__mouse_marker_indexes(
        "[100]", mouse_to_nic_markers_data
    )
    start_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        start_marker_index_list
    ]
    stop_times_from_mouse_to_nic_markers = mouse_to_nic_markers_time[
        stop_marker_index_list
    ]

    return (
        start_times_from_mouse_to_nic_markers,
        stop_times_from_mouse_to_nic_markers,
    )


def get_start_stop_times_from_mouse_markers(mouse_markers):
    """get the start and stop of the mouse motion from Mouse markers"""

    mouse_markers_data = mouse_markers["time_series"]
    mouse_markers_time = mouse_markers["time_stamps"]

    start_marker_index_list = find__mouse_marker_indexes(
        "DoCycleChange:DoRecord", mouse_markers_data
    )
    stop_marker_index_list = find__mouse_marker_indexes(
        "DoCycleChange:DoPause", mouse_markers_data
    )
    start_times_from_mouse_markers = mouse_markers_time[start_marker_index_list]
    stop_times_from_mouse_markers = mouse_markers_time[stop_marker_index_list]

    return (
        start_times_from_mouse_markers,
        stop_times_from_mouse_markers,
    )

In [ ]:
event_markers = get_stream(xdf_data, "Markers", ["event_ide_TONIC"])
mouse_to_nic_markers = get_stream(xdf_data, "Markers", ["MouseToNIC"])

if mouse_to_nic_markers:
    mouse_to_nic_markers_data = mouse_to_nic_markers["time_series"]
    mouse_to_nic_markers_time = mouse_to_nic_markers["time_stamps"]
    start_t, stop_t = get_start_stop_times_from_mouse_to_nic_markers(
        mouse_to_nic_markers
    )

# To verify that we get the same start and stop times from the mouse markers and the mouse to nic markers
# mouse_markers = get_stream(xdf_data, "Markers", ["Mouse", "Mouse-Markers"])
# if mouse_markers:
#     mouse_markers_data = mouse_markers["time_series"]
#     mouse_markers_time = mouse_markers["time_stamps"]
#     start_t, stop_t = get_start_stop_times_from_mouse_markers(mouse_markers)


if kinect_mocap:
    kinect_t = kinect_mocap["time_stamps"]
    WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
    WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
    WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

    WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
    WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
    WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

    WristLeft_Norm = np.sqrt(WristLeft_X**2 + WristLeft_Y**2 + WristLeft_Z**2)
    WristRight_Norm = np.sqrt(WristRight_X**2 + WristRight_Y**2 + WristRight_Z**2)


if event_markers:
    event_markers_data = event_markers["time_series"]
    event_markers_time = event_markers["time_stamps"]
    print(f"Event markers data shape: {event_markers_data.shape}")
    print(f"Event markers time shape: {event_markers_time.shape}")

In [ ]:
def set_start_stop_list(start_t, stop_t):
    """Set the start and stop times to be the same length and to be in the same format"""

    start_t = np.array(start_t)
    stop_t = np.array(stop_t)

    # create an empty array like start_t to store the new stop times
    new_stop_t = np.zeros_like(stop_t) + np.nan

    for i in range(len(start_t)):
        # get the closest stop that is greater than the start
        i_next_stop = np.where(stop_t > start_t[i])[0]
        if len(i_next_stop) > 0:
            i_next_stop = i_next_stop[0]
            new_stop_t[i] = stop_t[i_next_stop]
    return [start_t, new_stop_t]


#
def get_min_norm_by_start_stop_block(
    start_stop, kinect_t, WristLeft_Norm, WristRight_Norm
):
    min_norm_list = []
    """From each start to stop block, get the minimal value of the norm of the wrist left and right"""

    for start_time, stop_time in zip(start_stop[0], start_stop[1]):
        # only if we are inside the kinect time
        if start_time >= kinect_t[0] and stop_time <= kinect_t[-1]:
            start_time_index = np.argmax(kinect_t >= start_time)
            time_mask = (kinect_t >= start_time) & (kinect_t <= stop_time)

            block_wrist_left_min_index = np.argmin(WristLeft_Norm[time_mask])
            block_wrist_right_min_index = np.argmin(WristRight_Norm[time_mask])

            block_wrist_right_min_index += start_time_index  # index in the kinect time
            block_wrist_left_min_index += start_time_index

            min_norm_list.append(
                {
                    "start_time": start_time,
                    "stop_time": stop_time,
                    "block_wrist_left_min_index": block_wrist_left_min_index,
                    "block_wrist_right_min_index": block_wrist_right_min_index,
                }
            )

    return min_norm_list


start_stop = set_start_stop_list(start_t, stop_t)

# print("Start and stop times:")
# for start_t__, stop_t__ in zip(start_stop[0], start_stop[1]):
#     print(f"{start_t__:5.2f} -> {stop_t__:5.2f}, {stop_t__ - start_t__:8.2f} s")

min_norm_list = get_min_norm_by_start_stop_block(
    start_stop, kinect_t, WristLeft_Norm, WristRight_Norm
)

In [ ]:
def span_select(xmin, xmax):
    indmin, indmax = np.searchsorted(kinect_t, (xmin, xmax))
    indmax = min(len(kinect_t) - 1, indmax)

    region_x = kinect_t[indmin:indmax]

    if len(region_x) >= 2:
        print(f"Selected region: {region_x[0]:3.2f} --> {region_x[-1]:3.2f}")
        time_zone["start"] = region_x[0]
        time_zone["stop"] = region_x[-1]
    else:
        print("Selected region is too small")


# time_zone is the global variable that will be used to store the selected time zone
time_zone = {
    "start": -1,
    "stop": -1,
}


def plot_wrists_and_start_stop(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Plot the wrists norms with the start and stop times"""

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(kinect_t, WristLeft_Norm, label="Wrist Left Norm", color="b")
    ax.plot(kinect_t, WristRight_Norm, label="Wrist Right Norm", color="k")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Distance from the kinect (m)")
    ax.legend()

    # add the start and stop times as vertical lines
    for start_t__ in start_t:
        plt.axvline(x=start_t__, color="g", linestyle="--", label="Start Time")
    for stop_t__ in stop_t:
        plt.axvline(x=stop_t__, color="r", linestyle="--", label="Stop Time")

    return fig, ax


def get_calibration_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Get the calibration time zone from the selected region"""

    fig, ax = plot_wrists_and_start_stop(
        kinect_t,
        WristLeft_Norm,
        WristRight_Norm,
        start_t,
        stop_t,
    )

    # get the current x-axis limits (before the rectangle selector is created)
    xlim = ax.get_xlim()
    # Create a SpanSelector instance
    span_selector = SpanSelector(
        ax,
        span_select,
        "horizontal",
        useblit=True,
        props=dict(alpha=0.5, facecolor="tab:pink"),
        interactive=True,
        drag_from_anywhere=True,
    )
    ax.set_title(
        "Select a region to use for the calibration... close the plot when done."
    )

    # Set the x-axis limits to the current limits
    ax.set_xlim(xlim)
    # Set the title and show the plot
    plt.show(block=True)
    return time_zone


def get_reaching_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
):
    """Get the reaching time zone from the selected region"""

    fig, ax = plot_wrists_and_start_stop(
        kinect_t,
        WristLeft_Norm,
        WristRight_Norm,
        start_t,
        stop_t,
    )

    # get the current x-axis limits (before the rectangle selector is created)
    xlim = ax.get_xlim()
    # Create a SpanSelector instance
    span_selector = SpanSelector(
        ax,
        span_select,
        "horizontal",
        useblit=True,
        props=dict(alpha=0.5, facecolor="tab:pink"),
        interactive=True,
        drag_from_anywhere=True,
    )
    ax.set_title("Select a region to use for the reaching... close the plot when done.")

    # Set the x-axis limits to the current limits
    ax.set_xlim(xlim)
    # Set the title and show the plot
    plt.show(block=True)
    return time_zone


##################################################################
calibration_time_zone = get_calibration_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
)
print(f"Calibration start time: {calibration_time_zone['start']}")
print(f"Calibration stop time: {calibration_time_zone['stop']}")

reaching_time_zone = get_reaching_time_zone(
    kinect_t,
    WristLeft_Norm,
    WristRight_Norm,
    start_t,
    stop_t,
)
print(f"Reaching start time: {reaching_time_zone['start']}")
print(f"Reaching stop time: {reaching_time_zone['stop']}")